# 07. Interpretability: Feature–Class Associations in Explanations (Appendix E.6; Figure 8; Appendix Figure E5)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

Implements Appendix E.6 (Methods 2.5; Results 3.3). The 100 Exp 4 explanations with a valid design label were coded against a fixed schema of 26 binary cues in two passes (Appendix E.6.1):

* **First pass** (`exp4_feature_coding.csv`; prompt `exp4_coding_prompt.txt`; model and schema `exp4_coding_metadata.json`): five textual cues (`text_sad`, `text_anger`, `text_touched`, `text_pride`, `text_admire`) and eight nonverbal cues (`face_tears`, `face_frown`, `face_smile`, `face_neutral`, `vocal_subdued`, `vocal_excited`, `body_still`, `body_animated`). The same pass also produced seven physiological codes without direction (`phys_*`); these are superseded and are not used.
* **Second pass** (`exp4_physio_direction_coding.csv`; prompt `exp4_physio_direction_prompt.txt`): thirteen physiological cues with the direction the explanation states: `cheek_up/down`, `forehead_up/down`, `nose_up/down`, `face_temp_up_other/down_other`, `hr_up/down`, `hrv_up/down`, `phys_nodirection`.

Each explanation is linked to the class obtained by applying the mean-threshold rule to the participant's own Exp 4 scores (column `pred`). The coding steps call a language model and are not re-run here; the notebook starts from the released coding files.

* **E.6.2–E.6.3 Feature–class association.** association(f, c) = P(c | f appears) − P(c), where P(c) is the base rate of the model-predicted class and P(c | f) the share of explanations citing cue f that received class c; 95% intervals from 2,000 participant resamples; cues with zero occurrences are omitted. Importance(f) = occurrence rate(f) × Σ_c |association(f, c)| (Figure 8B–C).
* **E.6.4 Worked example** and the Fisher exact tests quoted in Results 3.3.
* **Figure 8A routing.** Cue citations by predicted class, the axis-level routing of the directional physiological cues with χ² tests across the four predicted classes, and the predicted × induced table.
* **E.6.3 step 5 and E.6.5 Valence separation by channel (Appendix Figure E5).** Positive minus negative cues per explanation in the textual and nonverbal channels, compared between the predicted Sympathy and Inspiration classes with Mann–Whitney *U*; for the physiological channel, the number of physiological cues cited per explanation.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load and merge the two coding passes

In [2]:
CODED = C.load_exp4_cue_table()
CUES_ALL = C.EXP4_TEXT_CUES + C.EXP4_NONVERBAL_CUES + C.EXP4_PHYSIO_CUES
CUES = [f for f in CUES_ALL if CODED[f].sum() > 0]           # E.6.3 step 3: omit cues with zero occurrences
CHANNEL = {f: C.exp4_channel(f) for f in CUES_ALL}
print("explanations:", len(CODED), "| cues in schema:", len(CUES_ALL), "(text", len(C.EXP4_TEXT_CUES), "/ nonverbal", len(C.EXP4_NONVERBAL_CUES), "/ physiological", len(C.EXP4_PHYSIO_CUES), ")")
print("cues never cited:", [f for f in CUES_ALL if f not in CUES])
print("predicted-class base rates:", CODED.pred.value_counts().reindex(C.CLS).to_dict())
CODED[CUES_ALL].sum().rename("explanations citing the cue").to_frame().T

explanations: 100 | cues in schema: 26 (text 5 / nonverbal 8 / physiological 13 )
cues never cited: ['face_tears', 'hr_down']
predicted-class base rates: {'ELVT': 29, 'SYMP': 25, 'INSP': 21, 'CTRL': 25}
Out[2]: 
                             text_sad  text_anger  text_touched  text_pride  text_admire  face_tears  face_frown  face_smile  face_neutral  vocal_subdued  \
explanations citing the cue        39          13            22          11           43           0           5          14            27              9   

                             vocal_excited  body_still  body_animated  cheek_up  cheek_down  forehead_up  forehead_down  nose_up  nose_down  \
explanations citing the cue              5           2              3        36          29           37             29        9         23   

                             face_temp_up_other  face_temp_down_other  hr_up  hr_down  hrv_up  hrv_down  phys_nodirection  
explanations citing the cue                   5               

## 2. E.6.2–E.6.3: association(f, c) with bootstrap intervals, and the importance score (Figure 8B–C)

In [3]:
def lifts(df):
    out = {}
    b = df.pred.value_counts(normalize=True)
    for f in CUES:
        sub = df[df[f] == 1]
        for c in C.CLS: out[(f, c)] = (sub.pred == c).mean() - b.get(c, 0.0) if len(sub) else np.nan
    return out
point = lifts(CODED); rng = np.random.default_rng(C.SEED); B = 2000
boot = {k: [] for k in point}
for _ in range(B):
    l = lifts(CODED.iloc[rng.integers(0, len(CODED), len(CODED))])
    for k in boot: boot[k].append(l[k])
rows = []
for f in CUES:
    for c in C.CLS:
        arr = np.array(boot[(f, c)], float); arr = arr[np.isfinite(arr)]
        rows.append(dict(feature=f, channel=CHANNEL[f], cls=c, n_present=int(CODED[f].sum()), lift=point[(f, c)], lo=np.percentile(arr, 2.5) if len(arr) else np.nan, hi=np.percentile(arr, 97.5) if len(arr) else np.nan))
LIFT = pd.DataFrame(rows); LIFT.to_csv(os.path.join(C.OUT, "exp4_feature_lift.csv"), index=False)
IMP = LIFT.groupby("feature").apply(lambda g: pd.Series(dict(sum_abs_lift=g.lift.abs().sum(), prevalence=g.n_present.iloc[0] / len(CODED))), include_groups=False).reset_index()
IMP["importance"] = IMP.sum_abs_lift * IMP.prevalence; IMP["channel"] = IMP.feature.map(CHANNEL); IMP = IMP.sort_values("importance", ascending=False)
IMP.to_csv(os.path.join(C.OUT, "exp4_feature_importance.csv"), index=False)
print(LIFT.pivot(index="feature", columns="cls", values="lift").loc[CUES, C.CLS].round(2).to_string())
IMP.round(3)

cls                   ELVT  SYMP  INSP  CTRL
feature                                     
text_sad              0.12  0.21 -0.18 -0.15
text_anger           -0.21  0.52 -0.21 -0.10
text_touched          0.39 -0.16 -0.03 -0.20
text_pride           -0.02 -0.25  0.34 -0.07
text_admire           0.15 -0.18  0.23 -0.20
face_frown           -0.09  0.55 -0.21 -0.25
face_smile           -0.08 -0.25  0.15  0.18
face_neutral         -0.10 -0.06 -0.02  0.19
vocal_subdued         0.04  0.19 -0.21 -0.03
vocal_excited        -0.09 -0.25  0.59 -0.25
body_still           -0.29  0.25 -0.21  0.25
body_animated        -0.29 -0.25  0.79 -0.25
cheek_up              0.15  0.06 -0.07 -0.14
cheek_down           -0.22 -0.11  0.17  0.16
forehead_up           0.12 -0.11  0.09 -0.09
forehead_down        -0.15  0.09 -0.11  0.16
nose_up              -0.07  0.08 -0.10  0.08
nose_down             0.06 -0.08  0.14 -0.12
face_temp_up_other    0.11 -0.05  0.19 -0.25
face_temp_down_other  0.14  0.18 -0.07 -0.25
hr_up     

## 3. E.6.4 Worked example and the Fisher exact tests of Results 3.3

Each test compares the share of explanations assigned to class c among those that cite the cue with the share among those that do not (2 × 2 table, two-sided). The tests are uncorrected for the number of cues.

In [4]:
from scipy import stats
L = LIFT.set_index(["feature", "cls"])
for f in ("text_touched", "hr_up"):
    sub = CODED[CODED[f] == 1]; print(f"{f}: cited in {len(sub)} explanations; by predicted class", sub.pred.value_counts().reindex(C.CLS, fill_value=0).to_dict(),
                                      "| association ELVT %+.3f, INSP %+.3f" % (L.loc[(f, "ELVT"), "lift"], L.loc[(f, "INSP"), "lift"]))
FISH = []
for feat, cls in [("text_touched", "ELVT"), ("cheek_down", "INSP"), ("cheek_up", "ELVT"), ("forehead_down", "CTRL"), ("hr_up", "ELVT"), ("text_anger", "SYMP"), ("vocal_excited", "INSP")]:
    a = int(((CODED[feat] == 1) & (CODED.pred == cls)).sum()); b = int(((CODED[feat] == 1) & (CODED.pred != cls)).sum())
    c_ = int(((CODED[feat] == 0) & (CODED.pred == cls)).sum()); d_ = int(((CODED[feat] == 0) & (CODED.pred != cls)).sum())
    OR, p = stats.fisher_exact([[a, b], [c_, d_]])
    FISH.append(dict(feature=feat, cls=cls, table=f"[[{a},{b}],[{c_},{d_}]]", association=L.loc[(feat, cls), "lift"], lo=L.loc[(feat, cls), "lo"], hi=L.loc[(feat, cls), "hi"], fisher_or=OR, p=p))
FISH = pd.DataFrame(FISH); print(FISH.round(3).to_string(index=False))
el = CODED.pred == "ELVT"
print(f"text_touched in {CODED.loc[el, 'text_touched'].mean():.3f} of Elevation explanations vs {CODED.loc[~el, 'text_touched'].mean():.3f} of the others")

text_touched: cited in 22 explanations; by predicted class {'ELVT': 15, 'SYMP': 2, 'INSP': 4, 'CTRL': 1} | association ELVT +0.392, INSP -0.028
hr_up: cited in 19 explanations; by predicted class {'ELVT': 10, 'SYMP': 3, 'INSP': 6, 'CTRL': 0} | association ELVT +0.236, INSP +0.106
      feature  cls             table  association    lo    hi  fisher_or     p
 text_touched ELVT  [[15,7],[14,64]]        0.392 0.219 0.562      9.796 0.000
   cheek_down INSP [[11,18],[10,61]]        0.169 0.029 0.320      3.728 0.014
     cheek_up ELVT [[16,20],[13,51]]        0.154 0.033 0.282      3.138 0.021
forehead_down CTRL [[12,17],[13,58]]        0.164 0.017 0.310      3.149 0.022
        hr_up ELVT  [[10,9],[19,62]]        0.236 0.036 0.447      3.626 0.022
   text_anger SYMP  [[10,3],[15,72]]        0.519 0.282 0.730     16.000 0.000
vocal_excited INSP   [[4,1],[17,78]]        0.590 0.093 0.850     18.353 0.007
text_touched in 0.517 of Elevation explanations vs 0.099 of the others


## 4. Figure 8A routing: cues by predicted class, axis routing of the directional physiological cues, and predicted × induced

In [5]:
CT = pd.DataFrame({f: CODED.loc[CODED[f] == 1, "pred"].value_counts().reindex(C.CLS, fill_value=0) for f in CUES}).T
CT["total"] = CT.sum(1); print(CT.to_string())
AXIS = []
for f, axis, level in [("cheek_up", "sympathy", 1), ("cheek_down", "sympathy", 0), ("forehead_up", "inspiration", 1), ("forehead_down", "inspiration", 0), ("hr_up", "inspiration", 1), ("hrv_up", "sympathy", 1)]:
    mp = C.SYM_HI if axis == "sympathy" else C.INS_HI; sub = CODED[CODED[f] == 1]
    AXIS.append(dict(feature=f, axis=axis, level="high" if level else "low", on_level=int((sub.pred.map(mp) == level).sum()), cited=len(sub)))
CHI = []
for f in [x for x in C.EXP4_PHYSIO_CUES if CODED[x].sum() > 0]:
    tab = pd.crosstab(CODED[f], CODED.pred).reindex(columns=C.CLS, fill_value=0); chi2, p, dof, _ = stats.chi2_contingency(tab.values)
    CHI.append(dict(feature=f, cited=int(CODED[f].sum()), chi2=chi2, df=dof, p=p))
AXIS = pd.DataFrame(AXIS); CHI = pd.DataFrame(CHI); print(AXIS.to_string(index=False)); print(CHI.round(4).to_string(index=False))
BOTH = {sig: int(((CODED[u] == 1) & (CODED[d] == 1)).sum()) for sig, u, d in [("cheek", "cheek_up", "cheek_down"), ("forehead", "forehead_up", "forehead_down"), ("nose/perioral", "nose_up", "nose_down"),
        ("other facial area", "face_temp_up_other", "face_temp_down_other"), ("heart rate", "hr_up", "hr_down"), ("HRV", "hrv_up", "hrv_down")]}
print("explanations citing both directions of the same signal:", BOTH)
PI = pd.crosstab(CODED.pred, CODED.true).reindex(index=C.CLS, columns=C.CLS, fill_value=0); print("rows = predicted, columns = induced:"); print(PI)
err = CODED[CODED.true != CODED.pred]; ins_err = err.true.map(C.INS_HI) != err.pred.map(C.INS_HI); sym_err = err.true.map(C.SYM_HI) != err.pred.map(C.SYM_HI)
print(f"correct {int((CODED.true == CODED.pred).sum())}; errors {len(err)}: inspiration axis {int(ins_err.sum())}, sympathy axis {int(sym_err.sum())}, both {int((ins_err & sym_err).sum())}")

pred                  ELVT  SYMP  INSP  CTRL  total
text_sad                16    18     1     4     39
text_anger               1    10     0     2     13
text_touched            15     2     4     1     22
text_pride               3     0     6     2     11
text_admire             19     3    19     2     43
face_frown               1     4     0     0      5
face_smile               3     0     5     6     14
face_neutral             5     5     5    12     27
vocal_subdued            3     4     0     2      9
vocal_excited            1     0     4     0      5
body_still               0     1     0     1      2
body_animated            0     0     3     0      3
cheek_up                16    11     5     4     36
cheek_down               2     4    11    12     29
forehead_up             15     5    11     6     37
forehead_down            4    10     3    12     29
nose_up                  2     3     1     3      9
nose_down                8     4     8     3     23
face_temp_up

## 5. Valence separation by channel (Appendix E.6.3 step 5; Appendix Figure E5) and cue counts (Appendix E.6.5)

Textual and nonverbal cues carry a valence (positive: moved, pride, praise, smile, excited voice, animated gestures; negative: sadness, anger, tears, frown, subdued voice, still posture). For the physiological channel the comparison uses the number of physiological cues cited per explanation, because the directional codes describe signals rather than valenced expressions.

In [6]:
POS = {"text": ["text_touched", "text_pride", "text_admire"], "nontext": ["face_smile", "vocal_excited", "body_animated"]}
NEG = {"text": ["text_sad", "text_anger"], "nontext": ["face_tears", "face_frown", "vocal_subdued", "body_still"]}
rows = []
for ch in POS:
    v = CODED[POS[ch]].sum(1) - CODED[NEG[ch]].sum(1); s = v[CODED.pred == "SYMP"]; i = v[CODED.pred == "INSP"]
    U, p = stats.mannwhitneyu(s, i, alternative="two-sided")
    rows.append(dict(channel=ch, measure="positive minus negative cues", mean_SYMP=s.mean(), mean_INSP=i.mean(), n_SYMP=len(s), n_INSP=len(i), U=U, p=p))
v = CODED[C.EXP4_PHYSIO_CUES].sum(1); s = v[CODED.pred == "SYMP"]; i = v[CODED.pred == "INSP"]; U, p = stats.mannwhitneyu(s, i, alternative="two-sided")
rows.append(dict(channel="physio", measure="physiological cues cited", mean_SYMP=s.mean(), mean_INSP=i.mean(), n_SYMP=len(s), n_INSP=len(i), U=U, p=p))
VAL = pd.DataFrame(rows); VAL.to_csv(os.path.join(C.OUT, "exp4_valence_by_channel.csv"), index=False); print(VAL.round(3).to_string(index=False))
print("E.6.5 counts (Sympathy / Inspiration explanations):", {f: (int(CT.loc[f, "SYMP"]), int(CT.loc[f, "INSP"])) for f in ["text_sad", "text_anger", "text_admire", "text_pride", "face_smile", "face_frown"]})

channel                      measure  mean_SYMP  mean_INSP  n_SYMP  n_INSP     U     p
   text positive minus negative cues      -0.92      1.333      25      21  27.5 0.000
nontext positive minus negative cues      -0.36      0.571      25      21 123.5 0.000
 physio     physiological cues cited       2.56      2.952      25      21 224.0 0.387
E.6.5 counts (Sympathy / Inspiration explanations): {'text_sad': (18, 1), 'text_anger': (10, 0), 'text_admire': (3, 19), 'text_pride': (0, 6), 'face_smile': (0, 5), 'face_frown': (4, 0)}


## 6. Verification

The thirteen textual and nonverbal cues were retained unchanged from the first coding pass, so their associations, intervals and importance scores, and the textual and nonverbal valence statistics, are compared with the frozen first-pass outputs (`exp4_feature_lift_predicted.csv`, `exp4_feature_importance_predicted.csv`, `exp4_valence_by_channel_predicted.csv`); the physiological rows of those files belong to the superseded undirected codes and are not compared. The directional physiological results have no frozen file and are compared with the values printed in Results 3.3, Figure 8, and Appendix E.6.4–E.6.5.

In [7]:
TN = C.EXP4_TEXT_CUES + C.EXP4_NONVERBAL_CUES
fl = C.load_frozen("exp4_feature_lift_predicted.csv"); fi_ = C.load_frozen("exp4_feature_importance_predicted.csv"); fv = C.load_frozen("exp4_valence_by_channel_predicted.csv")
print(C.compare(LIFT[LIFT.feature.isin(TN)], fl, ["feature", "cls"], ["n_present", "lift", "lo", "hi"], atol=0.05).to_string(index=False))
print(C.compare(IMP[IMP.feature.isin(TN)], fi_, ["feature"], ["sum_abs_lift", "prevalence", "importance"], atol=0.001).to_string(index=False))
print(C.compare(VAL[VAL.channel != "physio"], fv, ["channel"], ["mean_SYMP", "mean_INSP", "n_SYMP", "n_INSP", "U", "p"], atol=0.001).to_string(index=False))
MS = [  # (description, recomputed value, manuscript value, tolerance per element; bootstrap bounds .04, others half a unit of the printed digit)
    ("base rates ELVT/SYMP/INSP/CTRL", tuple(CODED.pred.value_counts().reindex(C.CLS)), (29, 25, 21, 25), 0),
    ("assoc moved->ELVT [lo, hi]", tuple(FISH.set_index("feature").loc["text_touched", ["association", "lo", "hi"]]), (.39, .21, .57), (.005, .04, .04)),
    ("assoc anger->SYMP [lo, hi]", tuple(FISH.set_index("feature").loc["text_anger", ["association", "lo", "hi"]]), (.52, .29, .75), (.005, .04, .04)),
    ("assoc excited voice->INSP [lo, hi]", tuple(FISH.set_index("feature").loc["vocal_excited", ["association", "lo", "hi"]]), (.59, .11, .85), (.005, .04, .04)),
    ("n excited voice", (int(CODED.vocal_excited.sum()),), (5,), 0),
    ("assoc cheek down->INSP [lo, hi], OR, p", tuple(FISH.set_index("feature").loc["cheek_down", ["association", "lo", "hi", "fisher_or", "p"]]), (.17, .03, .32, 3.7, .014), (.005, .04, .04, .05, .0005)),
    ("assoc cheek up->ELVT [lo, hi], OR, p", tuple(FISH.set_index("feature").loc["cheek_up", ["association", "lo", "hi", "fisher_or", "p"]]), (.15, .03, .28, 3.1, .021), (.005, .04, .04, .05, .0005)),
    ("assoc forehead down->CTRL [lo, hi], OR, p", tuple(FISH.set_index("feature").loc["forehead_down", ["association", "lo", "hi", "fisher_or", "p"]]), (.16, .02, .31, 3.1, .022), (.005, .04, .04, .05, .0005)),
    ("assoc HR up->ELVT [lo, hi], OR, p", tuple(FISH.set_index("feature").loc["hr_up", ["association", "lo", "hi", "fisher_or", "p"]]), (.24, .04, .45, 3.6, .022), (.005, .04, .04, .05, .0005)),
    ("E.6.4 moved: n, ELVT, association", (int(CT.loc["text_touched", "total"]), int(CT.loc["text_touched", "ELVT"]), L.loc[("text_touched", "ELVT"), "lift"]), (22, 15, .392), .0005),
    ("E.6.4 HR up: n, ELVT, INSP, assoc ELVT, assoc INSP", (int(CT.loc["hr_up", "total"]), int(CT.loc["hr_up", "ELVT"]), int(CT.loc["hr_up", "INSP"]), L.loc[("hr_up", "ELVT"), "lift"], L.loc[("hr_up", "INSP"), "lift"]), (19, 10, 6, .236, .106), .0005),
    ("moved: Fisher OR (ELVT); share in ELVT", (FISH.set_index("feature").loc["text_touched", "fisher_or"], CODED.loc[el, "text_touched"].mean()), (9.8, .52), (.05, .005)),
    ("moved: share in other classes (7 of 71)", (CODED.loc[~el, "text_touched"].mean(),), (.10,), .005),
    ("praise by class ELVT/SYMP/INSP/CTRL", tuple(CT.loc["text_admire", C.CLS]), (19, 3, 19, 2), 0),
    ("sad by class SYMP/ELVT", tuple(CT.loc["text_sad", ["SYMP", "ELVT"]]), (18, 16), 0),
    ("moved: ELVT of total", (int(CT.loc["text_touched", "ELVT"]), int(CT.loc["text_touched", "total"])), (15, 22), 0),
    ("anger: SYMP of total", (int(CT.loc["text_anger", "SYMP"]), int(CT.loc["text_anger", "total"])), (10, 13), 0),
    ("axis routing (on-level / cited), six cues", tuple(AXIS.on_level) + tuple(AXIS.cited), (27, 23, 26, 22, 16, 24, 36, 29, 37, 29, 19, 32), 0),
    ("HR up in Control", (int(CT.loc["hr_up", "CTRL"]),), (0,), 0),
    ("chi2 p: six routed cues max; others min", (CHI.set_index("feature").loc[list(AXIS.feature), "p"].max(), CHI.set_index("feature").loc[["nose_up", "nose_down", "hrv_down", "phys_nodirection"], "p"].min()), (.034, .14), (.0005, .005)),
    ("phys_nodirection n; hr_down n", (int(CODED.phys_nodirection.sum()), int(CODED.hr_down.sum())), (13, 0), 0),
    ("correct (ELVT/SYMP/INSP/CTRL)", (int((CODED.true == CODED.pred).sum()),) + tuple(int(PI.loc[c, c]) for c in C.CLS), (69, 17, 17, 15, 20), 0),
    ("absorbed: pred ELVT<-INSP, ELVT<-SYMP, SYMP<-ELVT, INSP<-CTRL", (int(PI.loc["ELVT", "INSP"]), int(PI.loc["ELVT", "SYMP"]), int(PI.loc["SYMP", "ELVT"]), int(PI.loc["INSP", "CTRL"])), (7, 4, 6, 5), 0),
    ("errors: inspiration axis, sympathy axis, both", (int(ins_err.sum()), int(sym_err.sum()), int((ins_err & sym_err).sum())), (19, 14, 2), 0),
    ("E.6.5 sad, anger in SYMP; praise, pride in INSP; smile INSP; frown SYMP", (int(CT.loc["text_sad", "SYMP"]), int(CT.loc["text_anger", "SYMP"]), int(CT.loc["text_admire", "INSP"]), int(CT.loc["text_pride", "INSP"]), int(CT.loc["face_smile", "INSP"]), int(CT.loc["face_frown", "SYMP"])), (18, 10, 19, 6, 5, 4), 0),
    ("E.6.5 physiological cues per explanation SYMP / INSP; p", tuple(VAL.set_index("channel").loc["physio", ["mean_SYMP", "mean_INSP", "p"]]), (2.56, 2.95, .39), (.005, .005, .005)),
    ("valence text / nonverbal p < .001", tuple(int(x < .001) for x in VAL.set_index("channel").loc[["text", "nontext"], "p"]), (1, 1), 0),
]
VER = pd.DataFrame([dict(item=d, recomputed=np.round(np.array(r, float), 3).tolist(), manuscript=list(m), match=bool(np.all(np.abs(np.array(r, float) - np.array(m, float)) <= np.array(t, float) + 1e-9))) for d, r, m, t in MS])
print(VER.to_string(index=False)); print("matching:", int(VER.match.sum()), "/", len(VER))
print("both directions of one signal cited (manuscript: none):", BOTH)

   column  n_compared  max_abs_diff  within_tolerance
n_present          48  0.000000e+00              True
     lift          48  9.020562e-17              True
       lo          48  2.478896e-02              True
       hi          48  3.791667e-02              True
      column  n_compared  max_abs_diff  within_tolerance
sum_abs_lift          12  5.551115e-17              True
  prevalence          12  0.000000e+00              True
  importance          12  9.020562e-17              True
   column  n_compared  max_abs_diff  within_tolerance
mean_SYMP           2  0.000000e+00              True
mean_INSP           2  0.000000e+00              True
   n_SYMP           2  0.000000e+00              True
   n_INSP           2  0.000000e+00              True
        U           2  0.000000e+00              True
        p           2  7.567731e-17              True
                                                                   item                                                     